In [ ]:
import os
import json
import cv2
import numpy as np
import pandas as pd


import mediapipe as mp
import torch
import torch.nn as nn


from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

In [2]:


DATASET_PATH = r"D:\sign language\video sentence\paul"
CSV_PATH = os.path.join(DATASET_PATH, "dataset.csv")
OUTPUT_PATH = r"D:\sign language\output"

os.makedirs(OUTPUT_PATH, exist_ok=True)

MAX_FRAMES = 64
BATCH_SIZE = 8
EPOCHS = 3
LEARNING_RATE = 0.0001

NUM_HANDS = 2
LANDMARKS = 21
FEATURES = NUM_HANDS * LANDMARKS * 3  # 126

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)

Device: cpu


In [3]:
def create_csv():
    records = []

    # WORD dataset
    word_path = os.path.join(DATASET_PATH, "word")

    if os.path.exists(word_path):
        for label in os.listdir(word_path):
            folder = os.path.join(word_path, label)

            if not os.path.isdir(folder):
                continue

            for file in os.listdir(folder):
                if file.lower().endswith((".jpg", ".jpeg", ".png")):
                    records.append([file, "word", os.path.join(folder, file), label])

    # SENTENCE dataset
    sentence_path = os.path.join(DATASET_PATH, "sentence")

    if os.path.exists(sentence_path):
        for label in os.listdir(sentence_path):
            label_folder = os.path.join(sentence_path, label)

            if not os.path.isdir(label_folder):
                continue

            for sample in os.listdir(label_folder):
                sample_folder = os.path.join(label_folder, sample)

                if not os.path.isdir(sample_folder):
                    continue

                images = [
                    f for f in os.listdir(sample_folder)
                    if f.lower().endswith((".jpg", ".jpeg", ".png"))
                ]

                if images:
                    records.append([sample, "sentence", sample_folder, label])

    if not records:
        raise ValueError("No dataset found. Check DATASET_PATH.")

    df = pd.DataFrame(records, columns=["name", "type", "path", "label"])
    df.to_csv(CSV_PATH, index=False)

    print("\nCSV created:", CSV_PATH)
    print("Samples:", len(df))
    print("Classes:", df["label"].nunique())

    return df


df = create_csv()


CSV created: D:\sign language\video sentence\paul\dataset.csv
Samples: 45
Classes: 6


In [4]:
# LABELS

labels = sorted(df["label"].unique())

label_to_id = {label: i for i, label in enumerate(labels)}
id_to_label = {i: label for label, i in label_to_id.items()}

df["label_id"] = df["label"].map(label_to_id)

NUM_CLASSES = len(labels)


In [5]:

with open(
    os.path.join(OUTPUT_PATH, "labels.json"),
    "w",
    encoding="utf-8"
) as f:
    json.dump(label_to_id, f, indent=4)

In [13]:
# 4. MEDIAPIPE FEATURE EXTRACTION

In [14]:
mp_hands = mp.solutions.hands

hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=NUM_HANDS,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)


In [15]:
def normalize_hand(hand):
    points = np.array(
        [[p.x, p.y, p.z] for p in hand.landmark],
        dtype=np.float32
    )

    # Wrist becomes origin
    points = points - points[0]

    # Scale
    scale = np.max(np.linalg.norm(points, axis=1))

    if scale > 0:
        points = points / scale

    return points.flatten()


In [16]:
def get_images(path, sample_type):
    if sample_type == "word":
        return [path]

    images = []

    if os.path.isdir(path):
        for file in os.listdir(path):
            if file.lower().endswith((".jpg", ".jpeg", ".png")):
                images.append(os.path.join(path, file))

    # Sort frame names
    images.sort()

    return images


In [17]:
def process_image(path):
    image = cv2.imread(path)

    if image is None:
        return np.zeros(FEATURES, dtype=np.float32)

    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    result = hands.process(image)

    features = np.zeros(FEATURES, dtype=np.float32)

    if not result.multi_hand_landmarks:
        return features

    left = None
    right = None

    for i, hand in enumerate(result.multi_hand_landmarks):

        data = normalize_hand(hand)

        side = result.multi_handedness[i].classification[0].label

        if side == "Left":
            left = data
        else:
            right = data

    if left is not None:
        features[:63] = left

    if right is not None:
        features[63:126] = right

    return features


In [ ]:
def extract_sample(path, sample_type):

    images = get_images(path, sample_type)

    if not images:
        return None

    # Keep maximum 120 frames
    if len(images) > MAX_FRAMES:
        indexes = np.linspace(
            0,
            len(images) - 1,
            MAX_FRAMES
        ).astype(int)

        images = [images[i] for i in indexes]

    features = []

    for image in images:
        features.append(process_image(image))

    features = np.array(features, dtype=np.float32)

    # Padding
    if len(features) < MAX_FRAMES:
        padding = np.zeros(
            (MAX_FRAMES - len(features), FEATURES),
            dtype=np.float32
        )

        features = np.vstack([features, padding])

    return features[:MAX_FRAMES]

In [ ]:
 
# 5. CREATE X AND Y
 

X = []
y = []

print("\nExtracting MediaPipe features...")

for _, row in df.iterrows():

    features = extract_sample(
        row["path"],
        row["type"]
    )

    if features is not None:
        X.append(features)
        y.append(row["label_id"])

X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=np.int64)

print("X shape:", X.shape)
print("Y shape:", y.shape)


np.save(
    os.path.join(OUTPUT_PATH, "X.npy"),
    X
)

np.save(
    os.path.join(OUTPUT_PATH, "y.npy"),
    y
)


Extracting MediaPipe features...
X shape: (45, 64, 126)
Y shape: (45,)


one hand  
21 * 3 = 64

two hand  
42 * 3 = 126

In [ ]:
 
# 6. TRAIN / VALIDATION SPLIT + DATALOADER
 

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y if len(np.unique(y)) > 1 else None
)


class SignDataset(Dataset):

    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, index):
        return self.X[index], self.y[index]


train_loader = DataLoader(
    SignDataset(X_train, y_train),
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    SignDataset(X_val, y_val),
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("\nTraining samples:", len(X_train))
print("Validation samples:", len(X_val))


Training samples: 36
Validation samples: 9


In [ ]:
 
# 7. SIMPLE TRANSFORMER
 

class SignTransformer(nn.Module):

    def __init__(self, num_classes):

        super().__init__()

        self.input_layer = nn.Linear(FEATURES, 128)

        encoder = nn.TransformerEncoderLayer(
            d_model=128,                     # all frame way to Transformer in  128 numbers .
            nhead=4,                         # 4 different "ways of looking for analyze
            dim_feedforward=256,             # 128 --> 256 expand feature 
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder,
            num_layers=2                        # 2 encoder layers 
        )

        self.classifier = nn.Linear(                # 256 ---> 1226 
            128,
            num_classes                                 
        )

    def forward(self, x):

        x = self.input_layer(x)

        x = self.transformer(x)

        # Average all frames
        x = x.mean(dim=1)

        return self.classifier(x)


model = SignTransformer(NUM_CLASSES).to(DEVICE)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)

print("\nModel created.")
print("Classes:", NUM_CLASSES)


Model created.
Classes: 6


In [ ]:
 
# 8. TRAINING
 

best_accuracy = 0

print("\nTraining started...\n")

for epoch in range(EPOCHS):

    # -------------------------
    # TRAIN
    # -------------------------

    model.train()

    train_loss = 0
    train_correct = 0

    for X_batch, y_batch in train_loader:

        X_batch = X_batch.to(DEVICE)
        y_batch = y_batch.to(DEVICE)

        optimizer.zero_grad()

        output = model(X_batch)

        loss = criterion(output, y_batch)

        loss.backward()

        optimizer.step()

        train_loss += loss.item()

        train_correct += (
            output.argmax(1) == y_batch
        ).sum().item()

    train_accuracy = train_correct / len(y_train)

    # -------------------------
    # VALIDATION
    # -------------------------

    model.eval()

    val_correct = 0
    val_loss = 0

    with torch.no_grad():

        for X_batch, y_batch in val_loader:

            X_batch = X_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)

            output = model(X_batch)

            loss = criterion(output, y_batch)

            val_loss += loss.item()

            val_correct += (
                output.argmax(1) == y_batch
            ).sum().item()

    val_accuracy = val_correct / len(y_val)

    print(
        f"Epoch {epoch + 1}/{EPOCHS} | "
        f"Train Acc: {train_accuracy * 100:.2f}% | "
        f"Val Acc: {val_accuracy * 100:.2f}%"
    )

    # Save best model
    if val_accuracy > best_accuracy:

        best_accuracy = val_accuracy

        torch.save(
            model.state_dict(),
            os.path.join(
                OUTPUT_PATH,
                "sign_transformer_best.pth"
            )
        )

        print("  Best model saved!")


Training started...

Epoch 1/3 | Train Acc: 22.22% | Val Acc: 33.33%
  Best model saved!
Epoch 2/3 | Train Acc: 38.89% | Val Acc: 33.33%
Epoch 3/3 | Train Acc: 44.44% | Val Acc: 33.33%


In [ ]:


torch.save(
    {
        "model": model.state_dict(),
        "classes": label_to_id,
        "num_classes": NUM_CLASSES
    },
    os.path.join(
        OUTPUT_PATH,
        "sign_transformer_final.pth"
    )
)

print("\nFinal model saved.")


Final model saved.


In [ ]:
 
# 10. TEST PREDICTION
 

model.eval()

sample = torch.tensor(
    X_val[0],
    dtype=torch.float32
).unsqueeze(0).to(DEVICE)

with torch.no_grad():

    output = model(sample)

    prediction = output.argmax(1).item()

print("\nTest prediction:")
print("Predicted:", id_to_label[prediction])
print("Actual:", id_to_label[y_val[0]])




Test prediction:
Predicted: DO
Actual: BED

Training complete!


what i mistake in this ? 

train  , test  --->>>> train , val , test 


testt is equal for avr  like 50 in each classes 


model save in 5 % ==0  on epochs 




face model  i have not use in this file .... 
